# `treasury(start, end=None, tenor=None)`

Daily US Treasury par yields by tenor, in percent, from FMP via `macro.treasury_rates`. Change the arguments and re-run any cell.

> **About the data shown.** This function's data comes from FMP's API, and FMP's license doesn't allow displaying it publicly. So the outputs saved here are synthetic: made-up data from [`_synthetic.py`](_synthetic.py), run through the function's own, unchanged code. The values are not real prices or yields.
>
> To see real data, start Jupyter with `ARORA_REAL_DATA=1`: every cell then reads prod, read-only, as `readonly_user`. Don't commit those outputs; `tests/test_notebooks_offline.py` fails if you save them.

In [1]:
# Synthetic data in place of FMP's (see the note above), unless Jupyter was started with ARORA_REAL_DATA=1.
import _synthetic

_synthetic.use()

Data: synthetic


In [2]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import treasury
from arora_data.treasury import TENORS, TreasuryRate


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The tenors
Twelve, from 1 month to 30 years, each stored as a column of `macro.treasury_rates`. A tenor works by its code in any case and spacing (`"10Y"`, `"10y"`, `"10 y"`) or by its column name (`"year10"`).

In [3]:
pl.DataFrame({"tenor": list(TENORS), "column": list(TENORS.values())})

tenor,column
str,str
"""1M""","""month1"""
"""2M""","""month2"""
"""3M""","""month3"""
"""6M""","""month6"""
"""1Y""","""year1"""
…,…
"""5Y""","""year5"""
"""7Y""","""year7"""
"""10Y""","""year10"""


## One tenor
`tenor="10Y"`: the 10-year yield by date. `yield_pct` is in percent, so 4.86 is 4.86%.

In [4]:
week = today_et() - timedelta(days=7)
assert treasury(week, tenor="10Y") == treasury(week, tenor="10y") == treasury(week, tenor="year10")
show(treasury(week, tenor="10Y"))

6 rows

tenor|date|yield_pct
---|---|---
10Y|2026-11-16|5.01
10Y|2026-11-17|4.98
10Y|2026-11-18|4.98
10Y|2026-11-19|4.96
10Y|2026-11-20|5.00
10Y|2026-11-23|5.01

## Every tenor, the default
Leave `tenor` out (or pass `"ALL"`) for the whole curve, ordered by maturity then date. Pivot it for one row per day.

In [5]:
curve = to_df(treasury(today_et() - timedelta(days=7)), TreasuryRate)
curve.pivot(on="tenor", index="date", values="yield_pct")

date,1M,2M,3M,6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
date,"decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]"
2026-11-16,3.91,3.94,3.98,4.07,4.23,4.48,4.64,4.84,4.94,5.01,5.10,5.13
2026-11-17,3.91,3.94,3.98,4.07,4.23,4.46,4.62,4.81,4.90,4.98,5.07,5.09
2026-11-18,3.91,3.94,3.98,4.07,4.23,4.46,4.62,4.80,4.90,4.98,5.06,5.09
2026-11-19,3.90,3.94,3.97,4.06,4.21,4.44,4.60,4.79,4.89,4.96,5.05,5.08
2026-11-20,3.97,4.00,4.03,4.12,4.27,4.49,4.65,4.83,4.92,5.00,5.08,5.11
2026-11-23,3.96,3.99,4.02,4.11,4.26,4.49,4.65,4.84,4.93,5.01,5.10,5.12


## The curve on one day, and a year earlier

In [6]:
last = curve["date"].max()
year_ago = to_df(treasury(last - timedelta(days=372), last - timedelta(days=365)), TreasuryRate)
year_ago_day = year_ago["date"].max()
(
    curve.filter(pl.col("date") == last).select("tenor", pl.col("yield_pct").alias(str(last)))
    .join(year_ago.filter(pl.col("date") == year_ago_day).select("tenor", pl.col("yield_pct").alias(str(year_ago_day))),
          on="tenor", how="left", maintain_order="left")
)

tenor,2026-11-23,2025-11-21
str,"decimal[38,2]","decimal[38,2]"
"""1M""",3.96,3.24
"""2M""",3.99,3.28
"""3M""",4.02,3.32
"""6M""",4.11,3.43
"""1Y""",4.26,3.63
…,…,…
"""5Y""",4.84,4.42
"""7Y""",4.93,4.59
"""10Y""",5.01,4.72


## Spreads come from two tenors
There is no spread argument: join two tenors. Here the 10-year minus the 2-year, the usual recession signal when it turns negative, over the last five years.

In [7]:
start = today_et() - timedelta(days=5 * 365)
ten, two = (to_df(treasury(start, tenor=t), TreasuryRate).select("date", pl.col("yield_pct").alias(t)) for t in ("10Y", "2Y"))
spread = ten.join(two, on="date").with_columns((pl.col("10Y") - pl.col("2Y")).alias("10Y-2Y"))
print(f"{spread.filter(pl.col('10Y-2Y') < 0).height} of {spread.height} days inverted")
spread.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True).agg(
    pl.col("10Y-2Y").min().alias("min"), pl.col("10Y-2Y").max().alias("max"), pl.col("10Y-2Y").last().alias("last"))

0 of 1304 days inverted


year,min,max,last
i32,"decimal[38,2]","decimal[38,2]","decimal[38,2]"
2021,0.63,0.72,0.72
2022,0.54,0.80,0.62
2023,0.44,0.82,0.79
2024,0.70,0.88,0.81
2025,0.67,1.13,0.68
2026,0.48,0.72,0.52


## Full history in one call
From 1990, the table's first day. There is no row limit. Each tenor's first day follows Treasury's own history: the 20-year from 1993, the 1-month from July 2001, the 2-month from 2018.

In [8]:
history = to_df(treasury(date(1900, 1, 1)), TreasuryRate)
print(f"{history.height:,} rows over {history['date'].n_unique():,} days, {history['date'].min()} to {history['date'].max()}")
history.group_by("tenor", maintain_order=True).agg(pl.col("date").min().alias("first"), pl.col("date").max().alias("last"), pl.len().alias("days"))

102,943 rows over 9,624 days, 1990-01-02 to 2026-11-23


tenor,first,last,days
str,date,date,u32
"""1M""",2001-07-31,2026-11-23,6604
"""2M""",2018-10-16,2026-11-23,2115
"""3M""",1990-01-02,2026-11-23,9624
"""6M""",1990-01-02,2026-11-23,9624
"""1Y""",1990-01-02,2026-11-23,9624
…,…,…,…
"""5Y""",1990-01-02,2026-11-23,9624
"""7Y""",1990-01-02,2026-11-23,9624
"""10Y""",1990-01-02,2026-11-23,9624


## Gaps are left out, not filled
A tenor with no value that day has no row. The 30-year was suspended from 2002-02-15 to 2006-02-09, and Columbus Day 2010 (10-11) is stored with no values at all, so it never appears.

In [9]:
thirty = history.filter(pl.col("tenor") == "30Y").with_columns(pl.col("date").shift().alias("previous"))
print(thirty.filter((pl.col("date") - pl.col("previous")).dt.total_days() > 7).select("previous", "date"))
show(treasury(date(2010, 10, 8), date(2010, 10, 12), tenor="10Y"))

shape: (1, 2)
┌────────────┬────────────┐
│ previous   ┆ date       │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2002-02-15 ┆ 2006-02-09 │
└────────────┴────────────┘


2 rows

tenor|date|yield_pct
---|---|---
10Y|2010-10-08|5.50
10Y|2010-10-12|5.46

## Bad input
A tenor that isn't one of the twelve, or a start after the end, raises a clear error.

In [10]:
for tenor in ["4M", "10", "10 years"]:
    try:
        treasury(week, tenor=tenor)
    except ValueError as err:
        print(f"{tenor}: {err}")
try:
    treasury(today_et(), week)
except ValueError as err:
    print(f"start after end: {err}")

4M: '4M' is not a tenor: use one of 1M, 2M, 3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 20Y, 30Y
10: '10' is not a tenor: use one of 1M, 2M, 3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 20Y, 30Y
10 years: '10 years' is not a tenor: use one of 1M, 2M, 3M, 6M, 1Y, 2Y, 3Y, 5Y, 7Y, 10Y, 20Y, 30Y
start after end: start must be on or before end
